# HDBSCAN & DBSCAN Analysis

* **DBSCAN** is a density based clustering algorithm. Instead of guessing how many clusters are needed, we define the hyperparameters ∈ and 𝑛.
The advantage of DBSCAN is that it can build clusters that have an arbitrary shape, while k-means and other centroid-based algorithms create clusters that have a shape of a hypersphere. The drawback is that it has 2 hyperparameters and choosing good values for them (especially ∈) can be challenging. Also, with a fixed ∈, the clustering algorithm cannot effectively deal with clusters of varying density.

* **HDBSCAN** has the same advantages as DBSCAN but only has 1 hyperparameter (𝑛). It can build clusters of varying density. 
  * 𝑛: the minimum number of examples to put in a cluster (simple to choose by intuition)

### References

[1] Andriy Burkov, *The Hundred-Page Machine Learning Book*, 2019.

[2] Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, 1st ed., O'Reilly Media, 2025.

[3] Scikit-learn Developers, "DBSCAN", Scikit-learn documentation:
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html

[4] Scikit-learn Developers, "HDBSCAN", Scikit-learn documentation:
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html

*Date: Jun 8, 2026 - By: Tomás Silva*

## 1. Imports and Configuration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## 2. Load 692799 I(q) sample data

In [8]:
file_path = 'Data/692799 IQ_fitting.csv'
df_day4 = pd.read_csv(file_path)
display(df_day4.head())

features = ['D_period', 'wp', 'area under third order curve', 'peak_width', 'secondmoment_lu']

# Initialize empty dictionaries
base_means = {}
base_stds = {}

print("--- Real Data Baselines (Day 4) ---\n")
for feature in features:
    mean_val = df_day4[feature].mean()
    std_val = df_day4[feature].std()
    
    # save on the dictionary
    base_means[feature] = mean_val
    base_stds[feature] = std_val
    
    print(f"Feature: {feature}")
    print(f"  Mean: {mean_val:.6f}")
    print(f"  Std Dev: {std_val:.6f}\n")

,x,y,total SAXS intensity,total_SAXS_norm_0_1,area under third order curve,collagen_third_norm_0_1,percentage_above_baseline,max_recorded_intensity,firstmoment,secondmoment,...,wMu,firstmoment_lu,secondmoment_lu,thirdmoment_lu,skewness_lu,D_period,peak_width,wp,peak_amplitude,fibril_radius
0,0,0,0.018897,0.004010,0.000000,1.000000e-09,34.733524,0.007163,0.281151,0.0791,...,0.187551,0.306105,0.0938,0.028730,2.599635,62.623109,0.001,0.056453,1,47.237054
1,1,0,0.019003,0.004349,0.000017,2.971487e-02,63.363078,0.005062,0.290088,0.0842,...,0.021837,0.292087,0.0853,0.024920,-0.144198,64.553274,0.001,0.006376,1,418.213630
2,2,0,0.018621,0.003124,0.000021,3.670660e-02,61.695855,0.004631,0.297568,0.0886,...,0.010000,0.282080,0.0796,0.022446,-0.662833,66.842397,0.001,0.002820,1,945.626478
3,3,0,0.019072,0.004572,0.000000,1.000000e-09,28.452648,0.006882,0.300850,0.0906,...,0.010000,0.282080,0.0796,0.022446,-0.662833,66.842397,0.001,0.002820,1,945.626478
4,4,0,0.019987,0.007506,0.000094,1.643057e-01,93.781777,0.005867,0.294687,0.0869,...,0.015918,0.297081,0.0883,0.026220,-0.503788,63.466518,0.001,0.004728,1,564.045008


--- Real Data Baselines (Day 4) ---

Feature: D_period
  Mean: 65.824251
  Std Dev: 1.510265

Feature: wp
  Mean: 0.023034
  Std Dev: 0.021533

Feature: area under third order curve
  Mean: 0.000118
  Std Dev: 0.000161

Feature: peak_width
  Mean: 0.003298
  Std Dev: 0.003104

Feature: secondmoment_lu
  Mean: 0.083125
  Std Dev: 0.003671



## 3. Generate Pseudo-Data with 5 sections

In [ ]:
# 1. Initialize empty grids (100 rows, 200 cols)
rows, cols = 100, 200
pseudo_dp = np.zeros((rows, cols))
pseudo_wp = np.zeros((rows, cols))
pseudo_area = np.zeros((rows, cols))
pseudo_pw = np.zeros((rows, cols))

# 2. Populate the grid with 5 WAVY zones
for r in range(rows):
    # Create sine wave borders
    bound_1 = int(40 + np.sin(r / 3.0) * 4)
    bound_2 = int(80 + np.sin(r / 12.0) * 9)
    bound_3 = int(120 + np.sin(r / 5.5) * 5)
    bound_4 = int(160 + np.sin(r / 12) * 12)

    for c in range(cols):
        # Determine the zone based on wavy boundaries
        if c < bound_1: shift = 0.0       # Zone 0: Healthy
        elif c < bound_2: shift = 1.0     # Zone 1: Mild
        elif c < bound_3: shift = 2.0     # Zone 2: Moderate
        elif c < bound_4: shift = 3.0     # Zone 3: Severe
        else: shift = 4.0                 # Zone 4: Core Wound

        # Apply values with random noise (0.25 * std)
        pseudo_dp[r, c] = np.random.normal(
            base_means['D_period'] + (shift * base_stds['D_period']), 
            base_stds['D_period'] * 0.25)
        
        pseudo_wp[r, c] = np.random.normal(
            base_means['wp'] + (shift * base_stds['wp']), 
            base_stds['wp'] * 0.25)
        
        pseudo_area[r, c] = np.random.normal(
            base_means['area under third order curve'] + (shift * base_stds['area under third order curve']), 
            base_stds['area under third order curve'] * 0.25)
        
        pseudo_pw[r, c] = np.random.normal(
            base_means['peak_width'] + (shift * base_stds['peak_width']), 
            base_stds['peak_width'] * 0.25)